# Merge & split datasets

Single pipeline that:
1. **separates** the 2025 questionnaire into its sub-samples (physics, high school, psychology, teacher, altro dipartimento),
2. **merges** them with the earlier cohorts into the cross-year *all-psychology* files and the *human* / *GPT* databases,
3. processes the **2019** experts / high-school data,
4. **splits** the relevant samples into low/high math-anxiety groups by the MAS-IT median.

The cue-words summary table lives in a separate notebook (`cue_words_summary.ipynb`).

In [1]:
import re
import numpy as np
import pandas as pd

from config import RESSERENO_ROOT, MATH_ANXIETY_ROOT
from questionnaires_analysis import low_high_anxiety_division

FORMATTING = RESSERENO_ROOT / "out" / "formatting"
MA_FORMATTING = MATH_ANXIETY_ROOT / "out" / "formatting"

## Load the source data

In [2]:
# 2025 questionnaire (every 2025 sub-sample lives in these two files)
mas_it_2025 = pd.read_csv(FORMATTING / "all_students_2025" / "mas_it_2025.csv")
assoc_2025 = pd.read_csv(
    FORMATTING / "all_students_2025" / "associations_valence_task_qualtrics.csv"
)

# earlier psychology cohorts (merged into the cross-year "all psychology" sample)
mas_it_2023 = pd.read_csv(MA_FORMATTING / "humans_2023" / "mas_it_2023_humans.csv")
mas_it_2024 = pd.read_csv(MA_FORMATTING / "humans_2024" / "mas_it_2024_humans.csv")
mas_it_BA_2024 = pd.read_csv(FORMATTING / "psychology_BA_2024" / "mas_it_2024.csv")

assoc_2023 = pd.read_csv(
    MA_FORMATTING / "humans_2023" / "associations_valence_2023_humans.csv"
)
assoc_2024 = pd.read_csv(
    MA_FORMATTING / "humans_2024" / "associations_valence_2024_humans.csv"
)
assoc_BA_2024 = pd.read_csv(
    FORMATTING / "psychology_BA_2024" / "associations_valence_2024.csv"
)

## Separate the 2025 sub-samples

Each 2025 sub-sample is identified by its `participant_id` prefix. `fisica` covers
`fisica` + `fisica_unitn` + `fisica_stem`; high school covers `liceo` + `liceo_sci` + `liceo_lin` (north) and `high_school_south`.

In [3]:
# 2025 sub-sample -> participant_id prefix
SAMPLES_2025 = {
    "physics": "fisica",
    "high_school": ("liceo", "high_school_south"),
    "psychology": "interfacce",  # feeds the cross-year all-psychology merge
    "teacher": "teacher",
    "altro_dipartimento": "altro",
}


def by_prefix(df, prefix):
    return df[df["participant_id"].astype(str).str.startswith(prefix)].reset_index(drop=True)


mas_it = {
    name: by_prefix(mas_it_2025, pref)
    for name, pref in SAMPLES_2025.items()
}
assoc = {
    name: by_prefix(assoc_2025, pref)
    for name, pref in SAMPLES_2025.items()
}

for name in SAMPLES_2025:
    print(f"{name:20s}: {len(mas_it[name]):3d} participants")

physics             :  41 participants
high_school         :  92 participants
psychology          :  23 participants
teacher             :   7 participants
altro_dipartimento  :   0 participants


In [4]:
# save the per-sample 2025 files (paths kept identical to the previous pipeline; teacher /
# altro dipartimento are new folders). psychology is only used in the merge below, so not saved here.
SAMPLE_OUTPUTS = {
    "physics": (
        "physics_2025",
        "mas_it_fisica_2025.csv",
        "associations_valence_fisica.csv",
    ),
    "high_school": (
        "liceo_2025",
        "mas_it_liceo_2025.csv",
        "associations_valence_liceo_2025.csv",
    ),
    "teacher": (
        "teacher_2025",
        "mas_it_teacher_2025.csv",
        "associations_valence_teacher_2025.csv",
    ),
    "altro_dipartimento": (
        "altro_dipartimento_2025",
        "mas_it_altro_2025.csv",
        "associations_valence_altro_2025.csv",
    ),
}

for name, (folder, mas_name, assoc_name) in SAMPLE_OUTPUTS.items():
    out_dir = FORMATTING / folder
    out_dir.mkdir(parents=True, exist_ok=True)
    mas_it[name].to_csv(out_dir / mas_name, index=False)
    assoc[name].to_csv(out_dir / assoc_name, index=False)

## Cross-year *all psychology* sample

In [5]:
# 2023 + 2024 + BA 2024 + 2025 psychology, aligned to the 2025 columns
cols_2025 = mas_it["psychology"].columns
psy_mas_cohorts = [mas_it_2024, mas_it_2023, mas_it_BA_2024, mas_it["psychology"]]
psy_assoc_cohorts = [assoc_2024, assoc_2023, assoc_BA_2024, assoc["psychology"]]

all_psychology_mas_it = pd.concat(
    [d[cols_2025.intersection(d.columns)] for d in psy_mas_cohorts], ignore_index=True
)
all_psychology_assoc = pd.concat(psy_assoc_cohorts, ignore_index=True)

psy_dir = FORMATTING / "all_psychology_students"
all_psychology_mas_it.to_csv(psy_dir / "mas_it_all_psy_students.csv", index=False)
all_psychology_assoc.to_csv(
    psy_dir / "associations_valence_all_psy_students.csv", index=False
)
print("all psychology students:", len(all_psychology_mas_it))

all psychology students: 316


## Human & GPT databases

Each sample's `participant_id`s are renumbered to `{sample}_001, {sample}_002, …` and the sub-samples
are concatenated into the merged `database/` files.

In [6]:
def renumber(mas_it_df, assoc_df, sample):
    """Renumber a sample's ids as '{sample}_NNN' and propagate the mapping to its associations."""
    mas_new = mas_it_df.reset_index(drop=True).copy()
    new_ids = [f"{sample}_{i:03d}" for i in range(1, len(mas_new) + 1)]
    id_map = dict(zip(mas_new["participant_id"], new_ids))
    mas_new["participant_id"] = new_ids

    assoc_new = assoc_df.copy()
    assoc_new["participant_id"] = assoc_df["participant_id"].map(id_map)
    return mas_new, assoc_new


def build_database(samples, out_prefix):
    """Renumber every sample and write the merged mas-it / associations database files."""
    mas_parts, assoc_parts = zip(
        *(
            renumber(mas_df, assoc_df, name)
            for name, (mas_df, assoc_df) in samples.items()
        )
    )
    all_mas = pd.concat(mas_parts, ignore_index=True)
    all_assoc = pd.concat(assoc_parts, ignore_index=True)

    db_dir = FORMATTING / "database"
    all_mas.to_csv(db_dir / f"{out_prefix}_mas_it.csv", index=False)
    all_assoc.to_csv(db_dir / f"{out_prefix}_associations_valence.csv", index=False)
    return all_mas, all_assoc

In [7]:
# human database: cross-year psychology + the 2025 samples
HUMAN_DB = {
    "psychology": (all_psychology_mas_it, all_psychology_assoc),
    "physics": (mas_it["physics"], assoc["physics"]),
    "high_school": (mas_it["high_school"], assoc["high_school"]),
    "teacher": (mas_it["teacher"], assoc["teacher"]),
    "altro_dipartimento": (mas_it["altro_dipartimento"], assoc["altro_dipartimento"]),
}
all_humans_mas_it, all_humans_assoc = build_database(HUMAN_DB, "all_humans")
print("all humans:", len(all_humans_mas_it))

all humans: 456


In [8]:
# GPT-oss database: personas generated in 2026 by src/gpt_oss_data_generation.py, one per human
# of the final sample, each answering the same cue list as its human counterpart
GPT_DIR = RESSERENO_ROOT / "data" / "gpt_oss_2026"
# personas built after the main run (src/gpt_oss_extra_personas.py), once they are simulated
GPT_EXTRA_DIR = RESSERENO_ROOT / "data" / "gpt_oss_2026_extra"


def read_gpt(name):
    files = [d / name for d in (GPT_DIR, GPT_EXTRA_DIR) if (d / name).exists()]
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


personas = read_gpt("personas.csv")
gpt_mas_it = read_gpt("mas_it_gpt_oss.csv")
gpt_assoc = read_gpt("associations_valence_gpt_oss.csv")

# the 10 fisica_unitn students joined the physics sample after the main run: seven are paired with a
# spare persona of their stratum (physics, Set 5a with ChatGPT) and gender, the other three have their own.
# the fisica_lecce spares were generated with a physics-student profile (before those humans turned out
# to be high school students), so they are physics personas like the others
UNITN_PERSONAS = {
    "fisica_unitn_002": "gpt_oss_fisica_033_spare",  # F
    "fisica_unitn_003": "gpt_oss_fisica_lecce_005_spare",  # F
    "fisica_unitn_004": "gpt_oss_fisica_034_spare",  # M
    "fisica_unitn_005": "gpt_oss_fisica_lecce_016_spare",  # M
    "fisica_unitn_006": "gpt_oss_fisica_lecce_038_spare",  # M
    "fisica_unitn_007": "gpt_oss_fisica_lecce_040_spare",  # M
    "fisica_unitn_008": "gpt_oss_fisica_unitn_008",  # M, from gpt_oss_2026_extra
    "fisica_unitn_009": "gpt_oss_fisica_unitn_009",  # M, from gpt_oss_2026_extra
    "fisica_unitn_011": "gpt_oss_fisica_unitn_011",  # M, from gpt_oss_2026_extra
    "fisica_unitn_012": "gpt_oss_fisica_lecce_012_spare",  # F
}
missing = set(UNITN_PERSONAS.values()) - set(gpt_mas_it["participant_id"])
if missing:
    print(f"not simulated yet, left out: {sorted(missing)}")
# a matched persona is kept only while its human is in the sample (the fisica_lecce personas were
# built as physics students, but those humans are high school students: high_school_south has its own)
human_ids = pd.concat(
    [mas_it["physics"], mas_it["high_school"], mas_it_BA_2024, mas_it["psychology"]]
)["participant_id"]
gpt_personas = personas[
    (~personas["is_spare"].astype(str).eq("True") & personas["human_id"].isin(human_ids))
    | personas["persona_id"].isin(UNITN_PERSONAS.values())
]
gpt_personas = gpt_personas[gpt_personas["persona_id"].isin(gpt_mas_it["participant_id"])]

GPT_DB = {}
for sample, ids in gpt_personas.groupby("sample")["persona_id"]:
    GPT_DB[f"gpt_oss_{sample}"] = (
        gpt_mas_it[gpt_mas_it["participant_id"].isin(ids)].reset_index(drop=True),
        gpt_assoc[gpt_assoc["participant_id"].isin(ids)].reset_index(drop=True),
    )
    print(f"gpt_oss_{sample:12s}: {len(ids):3d} personas")

all_gpt_mas_it, all_gpt_assoc = build_database(GPT_DB, "all_gpt")
print("all gpt:", len(all_gpt_mas_it))


not simulated yet, left out: ['gpt_oss_fisica_unitn_008', 'gpt_oss_fisica_unitn_009', 'gpt_oss_fisica_unitn_011']
gpt_oss_high_school :  62 personas
gpt_oss_physics     :  38 personas
gpt_oss_psychology  : 189 personas
all gpt: 289


## 2019 experts & high-school data

Parse the raw 2019 text files into the long `associations_valence_{sample}.csv` format used elsewhere.

In [9]:
DATA_2019 = RESSERENO_ROOT / "data" / "data_2019"


def cue_word_extraction(file):
    """Read a 2019 associations file into (cue_word, association) pairs + the cue-word list."""
    pairs = [re.findall(r'"([^"]+)"', line) for line in open(file).readlines()]
    df = pd.DataFrame(pairs, columns=["association_0", "association_1"])
    counts = df["association_0"].value_counts()
    cue_words = counts[: len(counts) // 2].index.tolist()
    return df, cue_words


def _assign_valence(row, col, valence_map):
    word = row[col]
    if word in valence_map and valence_map[word]:
        return float(valence_map[word].pop(0).replace(".", ""))
    return np.nan


def association_valence_df(valence_file, df, sample):
    """Attach the 2019 valence ratings to a pairs dataframe and reshape to long format."""
    rows = [
        v.strip().split("\t")
        for v in open(DATA_2019 / f"{valence_file}.txt").readlines()
    ]
    valence_map = {}
    for row in rows:
        if not row[0].replace(".", "").isdigit():
            valence_map.setdefault(row[0], []).extend(
                [v for v in row[1:] if v.replace(".", "").isdigit()]
            )

    df = df.copy()
    df["valence_association_1"] = df.apply(
        _assign_valence, axis=1, args=("association_1", valence_map)
    )
    df["valence_association_0"] = df.apply(
        _assign_valence, axis=1, args=("association_0", valence_map)
    )
    df["new_index"] = df.index + 1
    df["cue_word_original"] = df["association_0"]

    df_long = pd.wide_to_long(
        df,
        stubnames=["association", "valence_association"],
        i=["new_index", "cue_word_original"],
        j="index",
        sep="_",
        suffix=r"\d+",
    ).reset_index()
    df_long = df_long.rename(
        columns={
            "association": "word",
            "valence_association": "valence",
            "cue_word_original": "cue_word",
        }
    )
    for col in ["cue_word", "word"]:
        df_long[col] = (
            df_long[col]
            .astype(str)
            .str.lower()
            .str.replace(r"[\d\W_]+", " ", regex=True)
            .str.strip()
        )
    df_long = df_long.dropna()
    df_long = df_long[(df_long["word"] != "nan") & (df_long["word"] != "")]

    out_dir = FORMATTING / sample
    out_dir.mkdir(parents=True, exist_ok=True)
    df_long.to_csv(out_dir / f"associations_valence_{sample}.csv", index=False)
    return df_long

In [10]:
students_df, _ = cue_word_extraction(DATA_2019 / "Students_unFiltered.txt")
experts_df, _ = cue_word_extraction(DATA_2019 / "Experts_unfiltered.txt")

association_valence_df("ValenceStudentsUPD", students_df, "students_2019")
association_valence_df("ValenceExperts", experts_df, "experts_2019")

,new_index,cue_word,index,word,valence
0,1,island,0,island,3.0
1,1,island,1,sea,3.0
3,2,land rover,1,island,3.0
5,3,land rover,1,sea,5.0
6,4,island,0,island,3.0
...,...,...,...,...,...
6361,3181,herb,1,trimmer,3.0
6363,3182,herb,1,vegetables,5.0
6366,3184,collapse,0,collapse,3.0
6367,3184,collapse,1,destroy,2.0


## Split by MAS-IT median (low / high anxiety)

`low_high_anxiety_division` writes `high_anx.csv` / `low_anx.csv` into `out/splitted_dfs/{sample}/`.

`psychology_sets_4_5` keeps only the psychology students whose cue set included all four target
words (*esame*, *voto*, *ansia*, *benessere*): BSc years 1-3 (2024, Set 4) and the 2025 students
(Set 5b). The 2023 and 2024 `student_exp` cohorts (Sets 2 and 3) are left out.
Add or remove a line to change which samples are split.

In [11]:
# sample name (= splitted_dfs subfolder) -> (mas-it df, associations df)
psy_sets_4_5_mas_it = pd.concat(
    [mas_it_BA_2024[cols_2025.intersection(mas_it_BA_2024.columns)], mas_it["psychology"]],
    ignore_index=True,
)
psy_sets_4_5_assoc = pd.concat([assoc_BA_2024, assoc["psychology"]], ignore_index=True)

SPLIT_SAMPLES = {
    "all_psychology_students": (all_psychology_mas_it, all_psychology_assoc),
    "psychology_sets_4_5": (psy_sets_4_5_mas_it, psy_sets_4_5_assoc),
    "high_school_2025": (mas_it["high_school"], assoc["high_school"]),
    "physics_2025": (mas_it["physics"], assoc["physics"]),
    "psychology_gpt_oss": GPT_DB["gpt_oss_psychology"],
    "high_school_gpt_oss": GPT_DB["gpt_oss_high_school"],
    "physics_gpt_oss": GPT_DB["gpt_oss_physics"],
}

for sample, (mas_df, assoc_df) in SPLIT_SAMPLES.items():
    (RESSERENO_ROOT / "out" / "splitted_dfs" / sample).mkdir(parents=True, exist_ok=True)
    low_high_anxiety_division(mas_df, assoc_df, sample, RESSERENO_ROOT)


Results for all_psychology_students sample 

total median of the summed scores: 37.0
number of total students: 316
number of students with high anxiety: 150
number of students with low anxiety: 151
number of students to exclude: 15

Results for psychology_sets_4_5 sample 

total median of the summed scores: 36.0
number of total students: 189
number of students with high anxiety: 93
number of students with low anxiety: 91
number of students to exclude: 5

Results for high_school_2025 sample 

total median of the summed scores: 30.0
number of total students: 92
number of students with high anxiety: 44
number of students with low anxiety: 45
number of students to exclude: 3

Results for physics_2025 sample 

total median of the summed scores: 30.0
number of total students: 41
number of students with high anxiety: 20
number of students with low anxiety: 17
number of students to exclude: 4

Results for psychology_gpt_oss sample 

total median of the summed scores: 47.0
number of total stud

## Participants summary

In [12]:
with open(FORMATTING / "participants_rassereno.txt", "a") as f:
    f.write(
        "Number of participants for Rassereno article:\n"
        "Experts 2019: 59\n"
        "High school students 2019: 159\n"
        f"High school students 2025: {len(mas_it['high_school'])}\n"
        f"Physics students 2025: {len(mas_it['physics'])}\n"
        f"Teachers 2025: {len(mas_it['teacher'])}\n"
        f"All psychology students: {len(all_psychology_mas_it)} "
        f"(2025 psy students: {len(mas_it['psychology'])})\n"
    )